# Demo - Anthropic SDK: enforcing a JSON schema

Structured outputs constrain the response to a JSON Schema you supply, so the reply is guaranteed to parse and guaranteed to match the shape you asked for.

| the schema guarantees | the schema does not guarantee |
|---|---|
| valid JSON | true values |
| every required field present | that a present field was *stated* |
| every enum value from your list | that your list covered reality |

## Setup

Install the SDK, load the API key from `.env`, and pin the model.

In [ ]:
!uv pip install anthropic dotenv

In [ ]:
import json

import anthropic
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"

## The tickets

Four support tickets, each one engineered to press on a different part of the schema.

| # | ticket | presses on |
|---|---|---|
| 1 | double charge, explicitly urgent, amount named | nothing — the control |
| 2 | a commercial partnership enquiry | the **category** enum: no member fits |
| 3 | a neutral question about delivery windows | **priority**: never stated |
| 4 | an angry complaint with no figure mentioned | **refund_amount**: no value exists |

In [ ]:
TICKETS = {
    "1 · double charge": (
        "I was charged twice for my March subscription - two identical $49.99 "
        "charges on the 3rd. I need one of them refunded. This is the third time "
        "this has happened and I am about to cancel, so please treat it as urgent."
    ),
    "2 · partnership": (
        "Hi - I run a logistics company in Melbourne and we are exploring a "
        "commercial partnership with your organisation. Who on your side handles "
        "reseller agreements? Happy to sign an NDA before we discuss numbers."
    ),
    "3 · delivery windows": (
        "Quick question about delivery windows. If I place an order on a Friday "
        "afternoon, does it dispatch that same day or the following Monday? Just "
        "trying to understand the schedule."
    ),
    "4 · damaged twice": (
        "This is completely unacceptable. My order arrived damaged, the replacement "
        "arrived damaged as well, and nobody has responded to either of my previous "
        "emails. I want this sorted out today."
    ),
}

## Reusable model call function

This code defines a reusable `extract` function which we can use to parse the different tickets into the JSON schema.

The JSON schema is specified using the `output_config` parameter:

- `output_config["format"]` is where the schema goes.
- `output_config["effort"]` is defined in the same dictionary - extraction doesn't need deep reasoning, so `low` is completely fine.

In [ ]:
SYSTEM = (
    "You are a support ticket triage system. Extract the requested fields from the "
    "ticket. Base every field strictly on what the ticket actually says."
)


def extract(ticket: str, schema: dict) -> dict:
    """Run one ticket through one schema and return the parsed object."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=SYSTEM,
        messages=[{"role": "user", "content": ticket}],
        output_config={
            "effort": "low",
            "format": {"type": "json_schema", "schema": schema},
        },
    )
    # With a format set, the first text block is guaranteed to be valid JSON.
    text = next(block.text for block in response.content if block.type == "text")
    return json.loads(text)

---

## 1 · The naive schema — no way to say "I don't know"

Features:
* three tight enums for `category` and `priority`
* a plain number
* every property is listed in `required`
* `additionalProperties` is `false`

In [ ]:
NAIVE_SCHEMA = {
    "type": "object",
    "properties": {
        "category": {
            "type": "string",
            "enum": ["billing", "shipping", "technical"],
            "description": "What the ticket is about.",
        },
        "priority": {
            "type": "string",
            "enum": ["low", "medium", "high"],
            "description": "How urgent the ticket is.",
        },
        "refund_amount": {
            "type": "number",
            "description": "The refund amount the customer is asking for, in dollars.",
        },
    },
    "required": ["category", "priority", "refund_amount"],
    "additionalProperties": False,
}

In [ ]:
naive_results = {}

for name, ticket in TICKETS.items():
    naive_results[name] = extract(ticket, NAIVE_SCHEMA)
    print(f"{name:<22} {naive_results[name]}")

Every input ticket parsed and matched to the schema, but three of the four are not correct:

- **Ticket 2** is a partnership enquiry. It is not billing, not shipping, not technical but the enum offers nothing else, so it ends up in one of the three regardless.
- **Ticket 3** never says how urgent it is, but `priority` comes back anyway as `low`
- **Tickets 2, 3 and 4** name no dollar figure. `refund_amount` is a `number` so a number is what you get — often `0` which is not "unknown", it's a claim that the
  customer asked for nothing

We provide no "escape hatch" if the ticket doesn't exactly match the schema's definitions.

---

## 2 · The schema with escape hatches

Three changes to the schema:

1. **A nullable field** — `refund_amount` becomes a `number` **or** `null` (for a value that may genuinely not exist)
2. **An `other` enum member, plus a companion field** — `other` alone tells you the enum failed but by itself throws away what the ticket was actually about. Pairing it with a nullable `category_other` keeps the information.
3. **A sentinel enum member** — `not_stated` on `priority`.

Note that 1 and 3 are **not** the same thing. `null` means *no value exists* whereas `not_stated` means *a value exists, but this document doesn't give it*. Every ticket has some real urgency; ticket 3 just doesn't tell you what it is.

In [ ]:
OPEN_SCHEMA = {
    "type": "object",
    "properties": {
        "category": {
            "type": "string",
            "enum": ["billing", "shipping", "technical", "other"],
            "description": (
                "What the ticket is about. Use 'other' if none of the named "
                "categories fit."
            ),
        },
        "category_other": {
            "anyOf": [{"type": "string"}, {"type": "null"}],
            "description": (
                "If category is 'other', a short description of the actual topic. "
                "Otherwise null."
            ),
        },
        "priority": {
            "type": "string",
            "enum": ["low", "medium", "high", "not_stated"],
            "description": (
                "How urgent the ticket is. Use 'not_stated' if the ticket gives no "
                "indication either way."
            ),
        },
        "refund_amount": {
            "type": ["number", "null"],
            "description": (
                "The refund amount the customer is asking for, in dollars. Null if "
                "the ticket names no amount."
            ),
        },
    },
    "required": ["category", "category_other", "priority", "refund_amount"],
    "additionalProperties": False,
}

In [ ]:
open_results = {}

for name, ticket in TICKETS.items():
    open_results[name] = extract(ticket, OPEN_SCHEMA)
    print(f"{name:<22} {open_results[name]}")

---

## 3 · Side by side comparison

In [ ]:
for name in TICKETS:
    naive, open_ = naive_results[name], open_results[name]
    print(name)
    print(
        f"   naive   category={str(naive['category']):<10} "
        f"priority={str(naive['priority']):<12} "
        f"refund={naive['refund_amount']}"
    )
    print(
        f"   open    category={str(open_['category']):<10} "
        f"priority={str(open_['priority']):<12} "
        f"refund={open_['refund_amount']}"
    )
    if open_["category_other"]:
        print(f"           category_other={open_['category_other']!r}")
    print()

---

## Takeaway

**Both runs were schema-valid.** Not one response failed validation, because there was never anything malformed to catch. The naive schema didn't break — it worked perfectly and faithfully recorded values the model had to make up.

Schema enforcement is a guarantee about *shape*. It moves the failure mode from "unparseable response" to "well-formed wrong answer" which is harder to notice and
travels much further downstream.

Three patterns for giving the model somewhere honest to put the gap:

| pattern | JSON Schema | use when |
|---|---|---|
| **Nullable field** | `{"type": "...", "null"}]` | the value may genuinely not exist |
| **`other` + companion** | extra enum member + nullable string sibling | your enum can't be exhaustive |
| **Sentinel member** | `"not_stated"` in the enum | a value exists but the source is silent |